In [1]:
import wandb
import pandas as pd
import numpy as np


group_names = [
    "CV-0-TransformerSymile-UKB-Intersect-Proteomics-512",
    "CV-1-TransformerSymile-UKB-Intersect-Proteomics-512",
    "CV-2-TransformerSymile-UKB-Intersect-Proteomics-512",
    "CV-3-TransformerSymile-UKB-Intersect-Proteomics-512",
    "CV-4-TransformerSymile-UKB-Intersect-Proteomics-512",
]

entity = "cardiors"
project = "gatedsymile"
modelname = "TransformerSymile"

api = wandb.Api()

metric = "test/acc_top1"

rows = []
for group in group_names:
    # get runs in this group
    runs = api.runs(f"{entity}/{project}", {"group": group})

    # parse split/method from the group string
    parts = group.split("-")
    split = parts[1]          # "0", "1", "2"
    method = parts[2]         # "GatedSymile"
    split = f"CV-{split}"     # make it "CV-0" etc.

    for run in runs:
        summary = dict(run.summary)

        # put the metric into a column named exactly like `metric`
        rows.append({
            "id": run.id,
            "name": run.name,
            "group": run.group,
            "state": run.state,
            "method": method,
            "split": split,
            metric: summary.get(metric, np.nan),
        })

df = pd.DataFrame(rows)

# optional: keep only finished runs
# df = df[df["state"].isin(["finished"])].copy()

print("rows:", len(df))
print(df)
print(df[["split", "method", metric]].head())

metric = "test/acc_top1"

# 1) aggregate seeds within (method, split)
agg = (df.dropna(subset=[metric])
         .groupby(["method", "split"])[metric]
         .agg(["mean", "count"])
         .reset_index())

# optional sanity check: expect 3 seeds per cell
print(agg.pivot(index="split", columns="method", values="count"))

# 2) wide table: one row per split, columns=methods
wide = agg.pivot(index="split", columns="method", values="mean").sort_index()
display(wide)

# Method-wise mean +/- stderr
for m in [modelname]:
    x = wide[m].dropna().values.astype(float)
    K = x.size
    mean_x = x.mean()
    stderr_x = x.std(ddof=1) / np.sqrt(K)
    print(f"{m}: {mean_x:.6f} ± {stderr_x:.6f} (stderr, K={K})")

rows: 15
          id                  name  \
0   w5b7rd16        lyric-wood-343   
1   8vk4iqt8     bright-shadow-341   
2   iuv9zal8        true-cloud-340   
3   csxq0bm4     driven-vortex-345   
4   5tnl5ufm      genial-water-344   
5   t549ahck  stellar-serenity-342   
6   7tq2nb50   peach-waterfall-348   
7   wer3cpg3      earnest-wood-347   
8   nshkhfmv     peachy-durian-346   
9   94vp3m0j       bright-pine-351   
10  0trnluii       easy-energy-350   
11  dkislj27      elated-field-349   
12  6nplbz4u       spring-dust-354   
13  xgro16zb        zany-smoke-353   
14  8mljokvl     silvery-grass-352   

                                                group     state  \
0   CV-0-TransformerSymile-UKB-Intersect-Proteomic...  finished   
1   CV-0-TransformerSymile-UKB-Intersect-Proteomic...  finished   
2   CV-0-TransformerSymile-UKB-Intersect-Proteomic...  finished   
3   CV-1-TransformerSymile-UKB-Intersect-Proteomic...  finished   
4   CV-1-TransformerSymile-UKB-Intersect-Proteo

method,TransformerSymile
split,
CV-0,0.555827
CV-1,0.641493
CV-2,0.664062
CV-3,0.624457
CV-4,0.658095


TransformerSymile: 0.628787 ± 0.019502 (stderr, K=5)
